# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset DOI: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The entities are referenced by their `@id` values.

In [ ]:
# List all record sets
record_sets = metadata.record_sets
if not record_sets:
    print('No record sets declared in the metadata. Inferring from files...')
    # Try to infer record sets from available files.
    for dist in metadata.distribution:
        print(f"Distribution @id: {getattr(dist, '@id', None)}")
        print(f"  Name: {getattr(dist, 'name', None)}")
        print(f"  URL: {getattr(dist, 'content_url', None)}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Fields:")
        for field in rs.get('field', []):
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id')} | Name: {field.get('name')}")
            else:
                print(f"    - Field reference: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

This dataset does not declare explicit `recordSets`; we'll extract the primary table from the first distribution.

In [ ]:
# Typically, Croissant lists record sets. This dataset appears to have the main recordset in the main file.
df = None
distribution_ids = [getattr(d, '@id', None) for d in metadata.distribution]
dataframes = {}
for distribution_id in distribution_ids:
    try:
        records = list(dataset.records(record_set=distribution_id))
        df = pd.DataFrame(records)
        dataframes[distribution_id] = df
        print(f"Loaded DataFrame for record_set: {distribution_id}")
        print(df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {distribution_id}: {e}")

# Use the first table for further analysis
if distribution_ids:
    selected_record_set = distribution_ids[0]
else:
    selected_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this dataset, we will select the numeric field `MW_kDa` (Molecular Weight in kDa) for demonstration, **referencing it by its `@id` as given in the schema or column name as extracted**.

In [ ]:
# Choose a numeric field (`MW_kDa` is a typical protein Mass Spectrometry field)
import numpy as np
if df is not None:
    numeric_field = 'MW_kDa' if 'MW_kDa' in df.columns else df.select_dtypes(include=np.number).columns[0]
    print(f"Using field for numeric analysis: {numeric_field}")
    threshold = 60  # Example: only proteins with MW > 60 kDa
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    group_field_candidates = ['Description', 'Sample', 'Type']
    group_field = None
    for field in group_field_candidates:
        if field in df.columns:
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and inspected the FAIR² dataset using the `mlcroissant` library and explored the molecular weight distribution of proteins isolated from human mast cell extracellular vesicles.
- Filtering and normalization highlighted proteins with higher molecular weights, which may represent specific subsets in the mass spectrometry analysis.
- Visualizations such as histograms and boxplots can reveal distribution and grouping differences for further biological or experimental insights.

This notebook provides a starting point—continue to tailor the analysis to domain-specific questions and hypotheses.